# 10 — Non-Sekans Özellik Ablasyon Çalışması (Bireysel Sütun)

**Amaç**: Sekans dışı (OHE ve k-mer olmayan) her bir sütunu tek tek çıkararak
LightGBM F1 / AUC değişimini ölçmek.

**Kapsam**: Conservation, fizikokimyasal, mühendislik özellikleri ve ham veri kalıntıları.

**Model**: LightGBM (12-combo grid search, 3-fold CV)

**Bağlantı**: NB09 → sekans gruplarını kaldırdı (grup bazlı)  
Bu notebook → non-sekans sütunları tek tek kaldırıyor (bireysel bazlı)

In [1]:
# Cell 1: Imports & Config
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, DATA_PATH, PANELS_SINGLE, REPORTS_DIR
from src.features import prepare_data_v3_no_insil
from src.models import grid_search_lightgbm
from src.metrics import compute_all_metrics

ABLATION_DIR = os.path.join(PROJECT_ROOT, 'results', 'ablation_nonseq')
os.makedirs(ABLATION_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'Proje koku: {PROJECT_ROOT}')
print(f'Ablasyon sonuc dizini: {ABLATION_DIR}')

Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Ablasyon sonuc dizini: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\ablation_nonseq


In [2]:
# Cell 2: Veri Yukleme & Feature Engineering
df_raw = pd.read_csv(DATA_PATH)
print(f'Ham veri: {df_raw.shape}')

sig = df_raw['clinvar__sig'].str.lower().str.strip()
target_map = {
    'benign': 0, 'likely benign': 0,
    'pathogenic': 1, 'likely pathogenic': 1,
}
df_raw['target'] = sig.map(target_map)
df_raw = df_raw.dropna(subset=['target'])
df_raw['target'] = df_raw['target'].astype(int)

panel_series = df_raw['Panel'].copy()
df_v3 = prepare_data_v3_no_insil(df_raw)
df_v3['Panel'] = panel_series.values

print(f'FE sonrasi: {df_v3.shape}')

# Panel belirleme
TRAIN_PANELS = []
for p in PANELS_SINGLE:
    p_df = df_v3[df_v3['Panel'] == p]
    if p_df['target'].value_counts().min() >= 5:
        TRAIN_PANELS.append(p)
print(f'Egitim panelleri: {TRAIN_PANELS}')

Ham veri: (4287, 119)
  71 in-silico sutunu droplandi
  OHE (ayri alfabe): 48 feature
  K-mer DNA_11mer_Ref: 16 2-mer feature
  K-mer DNA_11mer_Alt: 16 2-mer feature
  K-mer Prot_11mer_Ref: 430 2-mer feature
  K-mer Prot_11mer_Alt: 434 2-mer feature
  24 gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 14 sutun dusuruldu
[V3-NoInSil] Veri boyutu: 4287 x 954
FE sonrasi: (4287, 954)
Egitim panelleri: ['General', 'Hereditary_Cancer']


In [3]:
# Cell 3: Non-Sekans Sutunlari Tespit Et

all_cols = df_v3.columns.tolist()

# Sekans prefixleri (NB09 ile ayni)
SEQ_PREFIXES = (
    'base__ref_base__',
    'base__alt_base__',
    'ref_amino__',
    'alt_amino__',
    'DNA_11mer_Ref__2mer_',
    'DNA_11mer_Alt__2mer_',
    'Prot_11mer_Ref__2mer_',
    'Prot_11mer_Alt__2mer_',
)

SKIP_COLS = {'target', 'Panel'}

def is_seq_col(c):
    return any(c.startswith(pfx) for pfx in SEQ_PREFIXES)

seq_cols    = [c for c in all_cols if is_seq_col(c)]
nonseq_cols = [c for c in all_cols if not is_seq_col(c) and c not in SKIP_COLS]

print(f'Toplam sutun  : {len(all_cols)}')
print(f'Sekans sutun  : {len(seq_cols)}')
print(f'Non-sekans    : {len(nonseq_cols)}')
print()
print('=== NON-SEKANS SUTUN LISTESI ===')
for i, c in enumerate(sorted(nonseq_cols)):
    print(f'  {i+1:3d}. {c}')

Toplam sutun  : 954
Sekans sutun  : 931
Non-sekans    : 21

=== NON-SEKANS SUTUN LISTESI ===
    1. blosum62_alt_self
    2. blosum62_delta
    3. blosum62_ref_self
    4. blosum62_score
    5. chirality_shift
    6. conservation_product
    7. delta_charge
    8. delta_hydropathy
    9. delta_pi
   10. delta_volume
   11. gerp__gerp_nr
   12. gerp__gerp_rs
   13. gerp_x_phylop
   14. gnomad4__af
   15. gnomad4__an
   16. gnomad4__nhomalt
   17. grantham_distance
   18. is_transition
   19. phastcons__phastcons100_vert
   20. phylop__phylop100_vert
   21. polarity_change


In [4]:
# Cell 4: Bireysel Sutun Ablasyon Dongusu

ablation_results = []

# Ablosyon listesi: once baseline, sonra her sutun
scenarios = [('__BASELINE__', None)] + [(c, c) for c in sorted(nonseq_cols)]

for panel in TRAIN_PANELS:
    print(f"\n{'='*70}")
    print(f"PANEL: {panel}  ({len(scenarios)-1} sutun + baseline = {len(scenarios)} senaryo)")
    print(f"{'='*70}")

    df_panel_full = df_v3[df_v3['Panel'] == panel].drop(columns=['Panel'])
    X_all = df_panel_full.drop(columns=['target'])
    y = df_panel_full['target']

    X_cv, X_holdout, y_cv, y_holdout = train_test_split(
        X_all, y, test_size=0.20, random_state=SEED, stratify=y
    )

    for i, (scenario_name, drop_col) in enumerate(scenarios):
        if drop_col is None:
            X_cv_s    = X_cv
            X_ho_s    = X_holdout
            n_dropped = 0
            display_name = 'Baseline (tum ozellikler)'
        else:
            cols_to_drop = [drop_col] if drop_col in X_all.columns else []
            X_cv_s    = X_cv.drop(columns=cols_to_drop)
            X_ho_s    = X_holdout.drop(columns=cols_to_drop)
            n_dropped = len(cols_to_drop)
            display_name = scenario_name

        t0 = time.time()
        try:
            model, best_combo, best_thr, y_prob_ho = grid_search_lightgbm(
                X_cv_s, y_cv, X_ho_s, y_holdout
            )
            y_pred_ho = (y_prob_ho >= best_thr).astype(int)
            metrics   = compute_all_metrics(y_holdout, y_pred_ho, y_prob_ho)
            elapsed   = time.time() - t0

            row = {
                'panel'       : panel,
                'sutun'       : display_name,
                'n_kaldirildi': n_dropped,
                'f1'          : metrics['f1'],
                'auc_roc'     : metrics['auc_roc'],
                'auc_pr'      : metrics['auc_pr'],
                'precision'   : metrics['precision'],
                'recall'      : metrics['recall'],
                'mcc'         : metrics['mcc'],
                'elapsed_sec' : round(elapsed, 1),
            }
        except Exception as e:
            row = {
                'panel': panel, 'sutun': display_name,
                'n_kaldirildi': n_dropped,
                'f1': np.nan, 'auc_roc': np.nan, 'auc_pr': np.nan,
                'precision': np.nan, 'recall': np.nan, 'mcc': np.nan,
                'elapsed_sec': round(time.time() - t0, 1),
            }
            print(f"  HATA [{display_name}]: {e}")

        ablation_results.append(row)

        tag = ' <-- BASELINE' if drop_col is None else ''
        print(f"  [{i+1:3d}/{len(scenarios)}] {display_name:<45s} "
              f"F1={row['f1']:.4f}  AUC={row['auc_roc']:.4f}  ({row['elapsed_sec']:.0f}s){tag}")

abl_df = pd.DataFrame(ablation_results)
abl_df.to_csv(os.path.join(ABLATION_DIR, 'ablation_nonseq_results.csv'), index=False)
print(f'\nSonuclar kaydedildi: {ABLATION_DIR}/ablation_nonseq_results.csv')


PANEL: General  (21 sutun + baseline = 22 senaryo)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.1} -> CV F1=0.9505
  [  1/22] Baseline (tum ozellikler)                     F1=0.9465  AUC=0.9674  (24s) <-- BASELINE
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.1} -> CV F1=0.9531
  [  2/22] blosum62_alt_self                             F1=0.9465  AUC=0.9671  (23s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.05} -> CV F1=0.9510
  [  3/22] blosum62_delta                                F1=0.9498  AUC=0.9693  (24s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 63, 'learning_rate': 0.05} -> CV F1=0.9506
  [  4/22] blosum62_ref_self                             F1=0.9493  AUC=0.9700  (23s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo:

In [5]:
# Cell 5: Delta Hesabi & Sonuc Tablosu
from IPython.display import display

abl_df = pd.read_csv(os.path.join(ABLATION_DIR, 'ablation_nonseq_results.csv'))

summary_rows = []

for panel in TRAIN_PANELS:
    panel_abl = abl_df[abl_df['panel'] == panel].copy()

    bl = panel_abl[panel_abl['sutun'] == 'Baseline (tum ozellikler)']
    bl_f1  = bl['f1'].values[0]
    bl_auc = bl['auc_roc'].values[0]

    panel_abl['delta_f1']  = panel_abl['f1']      - bl_f1
    panel_abl['delta_auc'] = panel_abl['auc_roc'] - bl_auc

    # Baseline'i ayir, non-baseline satirlari delta_f1'e gore sirala
    non_bl = panel_abl[panel_abl['sutun'] != 'Baseline (tum ozellikler)'].sort_values('delta_f1')

    print(f'\n=== {panel}  (Baseline F1={bl_f1:.4f}, AUC={bl_auc:.4f}) ===')
    print(f'\nEn cok zararlı 10 sutun (kaldirilinca F1 en fazla dusüyor):')
    display(non_bl.head(10)[['sutun','f1','delta_f1','auc_roc','delta_auc']].round(4).reset_index(drop=True))

    print(f'\nEn az etkili 10 sutun (F1 neredeyse degismiyor):')
    display(non_bl.tail(10)[['sutun','f1','delta_f1','auc_roc','delta_auc']].round(4).reset_index(drop=True))

    print(f'\nFaydalı olan sutunlar (kaldirilinca F1 artıyor, yani gurultu):')
    beneficial = non_bl[non_bl['delta_f1'] > 0.001]
    if len(beneficial) > 0:
        display(beneficial[['sutun','f1','delta_f1']].round(4).reset_index(drop=True))
    else:
        print('  Yok (tum sutunlar F1 uzerinde nötr veya pozitif katkili)')

    summary_rows.append(panel_abl)

summary_df = pd.concat(summary_rows, ignore_index=True)
summary_df.to_csv(os.path.join(ABLATION_DIR, 'ablation_nonseq_with_delta.csv'), index=False)
print(f'\nDelta tablosu kaydedildi: {ABLATION_DIR}/ablation_nonseq_with_delta.csv')


=== General  (Baseline F1=0.9465, AUC=0.9674) ===

En cok zararlı 10 sutun (kaldirilinca F1 en fazla dusüyor):


,sutun,f1,delta_f1,auc_roc,delta_auc
0,gnomad4__an,0.9385,-0.0081,0.9663,-0.0011
1,gnomad4__af,0.9388,-0.0078,0.9547,-0.0127
2,delta_charge,0.9439,-0.0026,0.9683,0.0009
3,gerp_x_phylop,0.9441,-0.0025,0.9696,0.0022
4,gerp__gerp_rs,0.9444,-0.0022,0.9681,0.0007
5,polarity_change,0.9444,-0.0022,0.9702,0.0028
6,gnomad4__nhomalt,0.9445,-0.0020,0.9672,-0.0002
7,gerp__gerp_nr,0.9451,-0.0014,0.9662,-0.0012
8,blosum62_score,0.9451,-0.0014,0.9704,0.0030
9,delta_pi,0.9456,-0.0009,0.9698,0.0024



En az etkili 10 sutun (F1 neredeyse degismiyor):


,sutun,f1,delta_f1,auc_roc,delta_auc
0,grantham_distance,0.9463,-0.0002,0.9685,0.0011
1,phastcons__phastcons100_vert,0.9464,-0.0001,0.9716,0.0041
2,is_transition,0.9465,0.0000,0.9697,0.0023
3,blosum62_alt_self,0.9465,0.0000,0.9671,-0.0003
4,delta_volume,0.9470,0.0005,0.9684,0.0010
5,conservation_product,0.9475,0.0009,0.9701,0.0027
6,phylop__phylop100_vert,0.9481,0.0015,0.9707,0.0033
7,delta_hydropathy,0.9487,0.0021,0.9694,0.0019
8,blosum62_ref_self,0.9493,0.0027,0.9700,0.0026
9,blosum62_delta,0.9498,0.0033,0.9693,0.0019



Faydalı olan sutunlar (kaldirilinca F1 artıyor, yani gurultu):


...


En az etkili 10 sutun (F1 neredeyse degismiyor):


,sutun,f1,delta_f1,auc_roc,delta_auc
0,gerp_x_phylop,0.9324,-0.0037,0.9834,0.0023
1,delta_charge,0.9343,-0.0019,0.9799,-0.0012
2,gerp__gerp_rs,0.9353,-0.0009,0.9767,-0.0043
3,is_transition,0.9362,0.0000,0.9828,0.0018
4,conservation_product,0.9371,0.0009,0.9771,-0.0039
5,gnomad4__nhomalt,0.9371,0.0009,0.9779,-0.0031
6,phastcons__phastcons100_vert,0.9379,0.0018,0.9804,-0.0006
7,phylop__phylop100_vert,0.9379,0.0018,0.9812,0.0002
8,delta_volume,0.9388,0.0026,0.9824,0.0014
9,delta_pi,0.9437,0.0075,0.9836,0.0025



Faydalı olan sutunlar (kaldirilinca F1 artıyor, yani gurultu):


,sutun,f1,delta_f1
0,phastcons__phastcons100_vert,0.9379,0.0018
1,phylop__phylop100_vert,0.9379,0.0018
2,delta_volume,0.9388,0.0026
3,delta_pi,0.9437,0.0075



Delta tablosu kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\ablation_nonseq/ablation_nonseq_with_delta.csv


In [6]:
# Cell 6: Gorsellestime
fig_paths = []

for panel in TRAIN_PANELS:
    panel_abl = summary_df[summary_df['panel'] == panel].copy()
    bl_f1 = panel_abl[panel_abl['sutun'] == 'Baseline (tum ozellikler)']['f1'].values[0]

    non_bl = panel_abl[panel_abl['sutun'] != 'Baseline (tum ozellikler)'].copy()
    non_bl = non_bl.sort_values('delta_f1')

    # --- 1. Tum sutunlar delta F1 bar chart ---
    colors = ['#F44336' if d < -0.0005 else '#FFC107' if abs(d) <= 0.0005 else '#4CAF50'
              for d in non_bl['delta_f1']]

    n_rows = len(non_bl)
    fig_h  = max(8, n_rows * 0.35)
    fig, ax = plt.subplots(figsize=(12, fig_h))
    bars = ax.barh(non_bl['sutun'], non_bl['delta_f1'], color=colors)
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=7.5)
    ax.set_title(f'{panel} - Non-Sekans Sutun Ablasyonu (dF1 vs Baseline={bl_f1:.4f})', fontsize=12)
    ax.set_xlabel('dF1 (negatif = sutun onemli, pozitif = gurultu)')
    ax.set_ylabel('Kaldirilan Sutun')
    # Renk aciklamasi
    from matplotlib.patches import Patch
    legend_items = [
        Patch(color='#F44336', label='Onemli (dF1 < -0.0005)'),
        Patch(color='#FFC107', label='Notr (|dF1| <= 0.0005)'),
        Patch(color='#4CAF50', label='Gurultu (dF1 > +0.0005)'),
    ]
    ax.legend(handles=legend_items, loc='lower right', fontsize=9)
    plt.tight_layout()
    p = os.path.join(ABLATION_DIR, f'{panel}_nonseq_delta_f1_all.png')
    fig.savefig(p, dpi=150)
    fig_paths.append(p)
    plt.show()

    # --- 2. Top-15 en etkili sutun (en fazla F1 dusuren) ---
    top15 = non_bl.head(15)  # en kucuk delta_f1 = en onemli
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    bars2 = ax2.barh(top15['sutun'], top15['delta_f1'],
                     color=['#F44336' if d < 0 else '#4CAF50' for d in top15['delta_f1']])
    ax2.axvline(0, color='black', linewidth=1)
    ax2.bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)
    ax2.set_title(f'{panel} - En Onemli 15 Non-Sekans Sutun', fontsize=12)
    ax2.set_xlabel('dF1')
    plt.tight_layout()
    p2 = os.path.join(ABLATION_DIR, f'{panel}_nonseq_top15.png')
    fig2.savefig(p2, dpi=150)
    fig_paths.append(p2)
    plt.show()

# --- 3. Panel karsilastirma: her sutun icin iki panel delta_f1 ---
if len(TRAIN_PANELS) > 1:
    pivot = summary_df[summary_df['sutun'] != 'Baseline (tum ozellikler)'].pivot(
        index='sutun', columns='panel', values='delta_f1'
    )
    # Sirala: en fazla ortalama negatif delta uste
    pivot['_mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('_mean').drop(columns=['_mean'])

    fig3, ax3 = plt.subplots(figsize=(10, max(8, len(pivot) * 0.38)))
    sns.heatmap(pivot.astype(float), annot=True, fmt='.4f', cmap='RdYlGn',
                center=0, ax=ax3, linewidths=0.4)
    ax3.set_title('Non-Sekans Ablasyon dF1 Heatmap (Panel x Sutun)', fontsize=12)
    ax3.set_xlabel('Panel')
    ax3.set_ylabel('Kaldirilan Sutun')
    plt.tight_layout()
    p3 = os.path.join(ABLATION_DIR, 'nonseq_heatmap.png')
    fig3.savefig(p3, dpi=150)
    fig_paths.append(p3)
    plt.show()

print(f'\nToplam {len(fig_paths)} grafik kaydedildi.')


Toplam 5 grafik kaydedildi.


In [7]:
# Cell 7: Grup Bazli Ozet (Conservation / Fizikokimyasal / Muhendislik / Diger)

# Sutunlari anlamli gruplara ata
def assign_group(col):
    c = col.lower()
    if any(k in c for k in ['gerp', 'phastcons', 'phylop']):
        return 'Conservation'
    if any(k in c for k in ['hydropathy', 'volume', 'delta_pi', 'delta_charge',
                             'polarity', 'chirality']):
        return 'Fizikokimyasal'
    if any(k in c for k in ['grantham', 'blosum', 'conservation_product',
                             'gerp_x', 'is_transition', 'is_cpg', 'gc_content']):
        return 'Muhendislik'
    return 'Ham_Veri'

print('=== GRUP BAZLI ABLASYON OZETI ===')

for panel in TRAIN_PANELS:
    panel_abl = summary_df[
        (summary_df['panel'] == panel) &
        (summary_df['sutun'] != 'Baseline (tum ozellikler)')
    ].copy()
    panel_abl['grup'] = panel_abl['sutun'].apply(assign_group)

    print(f'\nPanel: {panel}')
    print(f'{"Grup":<20} {"Sutun Sayisi":>12} {"Ort dF1":>10} {"Min dF1":>10} {"Max dF1":>10}')
    print('-' * 65)
    for grp, grp_df in panel_abl.groupby('grup'):
        print(f'{grp:<20} {len(grp_df):>12} '
              f'{grp_df["delta_f1"].mean():>+10.4f} '
              f'{grp_df["delta_f1"].min():>+10.4f} '
              f'{grp_df["delta_f1"].max():>+10.4f}')

    # En onemli 5 sutun (en buyuk negatif delta)
    top5 = panel_abl.nsmallest(5, 'delta_f1')
    print(f'  En onemli 5 sutun: {", ".join(top5["sutun"].tolist())}')

=== GRUP BAZLI ABLASYON OZETI ===

Panel: General
Grup                 Sutun Sayisi    Ort dF1    Min dF1    Max dF1
-----------------------------------------------------------------
Conservation                    5    -0.0009    -0.0025    +0.0015
Fizikokimyasal                  6    -0.0006    -0.0026    +0.0021
Ham_Veri                        3    -0.0060    -0.0081    -0.0020
Muhendislik                     7    +0.0008    -0.0014    +0.0033
  En onemli 5 sutun: gnomad4__an, gnomad4__af, delta_charge, gerp_x_phylop, gerp__gerp_rs

Panel: Hereditary_Cancer
Grup                 Sutun Sayisi    Ort dF1    Min dF1    Max dF1
-----------------------------------------------------------------
Conservation                    5    -0.0015    -0.0066    +0.0018
Fizikokimyasal                  6    -0.0013    -0.0056    +0.0075
Ham_Veri                        3    -0.0212    -0.0578    +0.0009
Muhendislik                     7    -0.0045    -0.0086    +0.0009
  En onemli 5 sutun: gnomad4__af

In [8]:
# Cell 8: PDF Rapor
from fpdf import FPDF
from datetime import datetime


class NonSeqReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'Teknofest - Non-Sekans Sutun Ablasyon Raporu', align='C',
                  new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', align='C')


pdf = NonSeqReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Baslik sayfasi
pdf.add_page()
pdf.set_font('Helvetica', 'B', 18)
pdf.ln(30)
pdf.cell(0, 14, 'Non-Sekans Sutun Ablasyon Raporu', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 12)
pdf.cell(0, 9, 'Bireysel sutun ablasyonu | Model: LightGBM | FE: v3_no_insil', align='C',
          new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 9, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}', align='C',
          new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 7, f'Analiz edilen non-sekans sutun sayisi: {len(nonseq_cols)}', align='C',
          new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 7, f'Panel: {", ".join(TRAIN_PANELS)}', align='C', new_x='LMARGIN', new_y='NEXT')

# Her panel icin sonuc tablosu
for panel in TRAIN_PANELS:
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 13)
    panel_abl = summary_df[summary_df['panel'] == panel].copy()
    bl_f1 = panel_abl[panel_abl['sutun'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    pdf.cell(0, 9, f'{panel} - Bireysel Ablasyon (Baseline F1={bl_f1:.4f})', new_x='LMARGIN', new_y='NEXT')

    # Sadece non-baseline satirlar, delta_f1'e gore sirali
    non_bl = panel_abl[panel_abl['sutun'] != 'Baseline (tum ozellikler)'].sort_values('delta_f1')

    col_w = [75, 18, 18, 18, 18, 18, 15]
    hdrs  = ['Kaldirilan Sutun', 'F1', 'dF1', 'AUC', 'dAUC', 'MCC', 'Sure']
    pdf.set_font('Helvetica', 'B', 8)
    for w, h in zip(col_w, hdrs):
        pdf.cell(w, 7, h, border=1, align='C')
    pdf.ln()

    pdf.set_font('Helvetica', '', 7)
    for _, row in non_bl.iterrows():
        vals = [
            str(row['sutun'])[:50],
            f"{row['f1']:.4f}",
            f"{row['delta_f1']:+.4f}",
            f"{row['auc_roc']:.4f}",
            f"{row['delta_auc']:+.4f}",
            f"{row['mcc']:.4f}",
            f"{row['elapsed_sec']:.0f}s",
        ]
        for w, v in zip(col_w, vals):
            pdf.cell(w, 6, v, border=1, align='C')
        pdf.ln()

# Grafikler
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        fname = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 11)
        pdf.cell(0, 9, fname, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.set_font('Helvetica', '', 9)
            pdf.cell(0, 7, f'Grafik yuklenemedi: {e}', new_x='LMARGIN', new_y='NEXT')

report_path = os.path.join(REPORTS_DIR, 'ablation_nonseq_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')

PDF rapor kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\ablation_nonseq_report.pdf
